# 04 Baseline Poisson Naïve Bayes Decoder (ThresholdCrossings, γ = 0)

This notebook implements the **baseline decoder** described in Issar et al. (2020),
*J Neurophysiol* 123:1472–1485.

## Context
The paper trains a neural network (NAS — Not A Sorter) that assigns each waveform
a spike-likelihood score Pspike ∈ [0,1].  A γ-threshold gates which waveforms are
used for decoding.  **γ = 0 means all threshold crossings are used** — this is the
baseline from which all NAS improvements are measured.

This notebook computes that baseline: a **Poisson Naïve Bayes decoder**
(paper §Materials and Methods, "Decoding") applied to the ThresholdCrossings
stream across all 312 sessions of Monkey N.

## Inputs
- `/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/session_master_index.csv`
- `/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index/decoder_trial_table.csv`
- `/kaggle/input/datasets/katakuricharlotte/03-stream-inventory-results/tables_stream_inventory/candidate_stream_manifest.csv`
- Raw NWB files: `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_baseline_decoder/session_baseline_accuracy.csv`
- `tables_baseline_decoder/decoder_unit_table.csv`
- `meta_baseline_decoder/baseline_decoder_report.json`
- `figures_baseline_decoder/fig01_*.{png,pdf}` … `fig07_*`

## Figures (paper-aligned)
| Figure | Paper analogue |
|---|---|
| fig01: Sample session accuracy report | Fig. 4A/B (γ = 0 baseline point) |
| fig02: Session-wise accuracy × trial count | Fig. 4A/B dual-axis |
| fig03: Accuracy vs days since first session | Fig. 5C |
| fig04: Early vs late session distributions | Fig. 5D |
| fig05: Session distribution histograms (4-panel) | Fig. 4C–F |
| fig06: Mean ± 1 SE normalised accuracy | Fig. 5E (γ = 0 slice) |
| fig07: Accuracy vs recording quality proxy | Fig. 5A/B |

In [ ]:
!pip install pynwb h5py

In [ ]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
from pynwb import NWBHDF5IO

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PAPER-STYLE GLOBAL SETTINGS — Issar et al. 2020 (J Neurophysiol)
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

C_BLACK   = "#1a1a1a"
C_MAROON  = "#8B2222"     # TC / primary accuracy
C_BLUE    = "#2166AC"     # baseline reference dot (paper Fig 4A)
C_GREEN   = "#1B7837"     # maximum / best accuracy
C_GRAY    = "#888888"     # chance level / control
C_ORANGE  = "#E08214"     # SpikingBandPower stream
C_EARLY   = "#2166AC"     # early sessions (blue, as in paper Fig 5E)
C_LATE    = "#8B2222"     # late sessions  (maroon, as in paper Fig 5E)

MARKER_TC  = "o"
MARKER_SBP = "s"
CHANCE     = 0.125        # 1/8 directions

def paper_axes(ax, top=True, right=True):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.85)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=top, right=right)

np.random.seed(42)

## Paths

In [ ]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")

IN_IDX_DIR    = Path("/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index")
IN_STREAM_DIR = Path("/kaggle/input/datasets/katakuricharlotte/03-stream-inventory-results/tables_stream_inventory")

SESSION_MASTER_CSV    = IN_IDX_DIR    / "session_master_index.csv"
DECODER_TRIAL_CSV     = IN_IDX_DIR    / "decoder_trial_table.csv"
CANDIDATE_STREAM_CSV  = IN_STREAM_DIR / "candidate_stream_manifest.csv"

OUT_FIG_DIR   = Path("/kaggle/working/figures_baseline_decoder")
OUT_TABLE_DIR = Path("/kaggle/working/tables_baseline_decoder")
OUT_META_DIR  = Path("/kaggle/working/meta_baseline_decoder")

for d in [OUT_FIG_DIR, OUT_TABLE_DIR, OUT_META_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SAMPLE_NWB = DATASET_DIR / "sub-Monkey-N_ses-20200127_ecephys.nwb"

print("DATASET_DIR     :", DATASET_DIR)
print("SESSION_MASTER  :", SESSION_MASTER_CSV)
print("DECODER_TRIALS  :", DECODER_TRIAL_CSV)
print("CANDIDATE_STREAM:", CANDIDATE_STREAM_CSV)
print("SAMPLE_NWB      :", SAMPLE_NWB)

In [ ]:
assert DATASET_DIR.exists(),           "Missing dataset directory"
assert SESSION_MASTER_CSV.exists(),    "Missing session master CSV (run notebook 02)"
assert DECODER_TRIAL_CSV.exists(),     "Missing decoder trial table (run notebook 02)"
assert CANDIDATE_STREAM_CSV.exists(),  "Missing candidate stream manifest (run notebook 03)"
assert SAMPLE_NWB.exists(),            "Missing sample NWB"

session_df  = pd.read_csv(SESSION_MASTER_CSV)
trial_df    = pd.read_csv(DECODER_TRIAL_CSV)
stream_df   = pd.read_csv(CANDIDATE_STREAM_CSV)

session_df["session_date"] = pd.to_datetime(session_df["session_date"], errors="coerce")
if "days_since_first_session" not in session_df.columns:
    d0 = session_df["session_date"].min()
    session_df["days_since_first_session"] = (session_df["session_date"] - d0).dt.days

tc_stream_df = stream_df[stream_df["name"] == "ThresholdCrossings"].copy().reset_index(drop=True)

print("session_df shape  :", session_df.shape)
print("trial_df shape    :", trial_df.shape)
print("stream_df shape   :", stream_df.shape)
print("TC manifest rows  :", len(tc_stream_df))
display(session_df.head(3))
display(trial_df.head(3))

## Helper functions

### Neural data extraction

For each session we load the `ThresholdCrossings` `ElectricalSeries` from the NWB file.
The stream shape is **(T × 96)**, where T is the number of samples.  We align samples to
trial timing using the `timestamps` array (if present) or the stream's `rate` and
`starting_time`.

### Decoding window

Following Issar et al. (2020) §Materials and Methods:
> *"Waveforms classified as spikes from 300 ms to 500 ms after fixation
> (i.e., 50 ms after stimulus offset) were used for decoding."*

The memory-guided saccade task timing is:
- Fixation: 200 ms → Stimulus flash: 50 ms → Delay: 500 ms
- Decoding window: [300 ms, 500 ms] relative to fixation onset

For the NWB trial table, `start_time` marks trial start (≈ fixation onset).
We extract TC counts in `[start + 0.30, start + 0.50]` seconds.

### Target label extraction

`mrs_target_position` stores (x, y) target coordinates.  We compute the polar angle
and discretise into **8 octants** (0°, 45°, 90°, …, 315°) for the center-out task.

In [ ]:
def safe_attr(obj, attr, default=None):
    try:
        return getattr(obj, attr)
    except Exception:
        return default

def get_tc_stream_meta(nwb_path):
    """Return (rate_hz, starting_time, has_timestamps) for ThresholdCrossings."""
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        for obj in nwb.all_children():
            if safe_attr(obj, "name") == "ThresholdCrossings":
                rate  = safe_attr(obj, "rate",          None)
                t0    = safe_attr(obj, "starting_time", 0.0)
                ts    = safe_attr(obj, "timestamps",    None)
                has_ts = ts is not None
                return rate, t0, has_ts
    return None, 0.0, False

def angle_to_octant(angle_deg):
    """Snap angle to nearest of 8 octants: 0,45,90,...,315."""
    octant = int((angle_deg + 22.5) % 360 // 45)
    return octant  # 0..7

def extract_target_labels(trials_df):
    """
    Discretise mrs_target_position (or index_target_position) into 8 direction labels.
    Returns a Series of integer labels 0–7, or NaN if unavailable.
    """
    # Try index_target_position first (already discrete in CO tasks)
    if "index_target_position" in trials_df.columns:
        labels = pd.to_numeric(trials_df["index_target_position"], errors="coerce")
        if labels.notna().mean() > 0.8 and labels.nunique() <= 12:
            return labels.astype("Int64")

    # Try mrs_target_position as (x, y) string or separate columns
    for col in ["mrs_target_position", "target_position"]:
        if col not in trials_df.columns:
            continue
        try:
            pos = trials_df[col].apply(
                lambda v: [float(x) for x in str(v).strip("[]() ").split()] if pd.notna(v) else [np.nan, np.nan]
            )
            x_vals = pos.apply(lambda p: p[0] if len(p) >= 1 else np.nan)
            y_vals = pos.apply(lambda p: p[1] if len(p) >= 2 else np.nan)
            angles = np.degrees(np.arctan2(y_vals, x_vals)) % 360
            return angles.apply(lambda a: angle_to_octant(a) if np.isfinite(a) else np.nan).astype("Int64")
        except Exception:
            continue

    return pd.Series([np.nan] * len(trials_df), dtype="Float64")

In [ ]:
def load_tc_data(nwb_path, rate_hz=None):
    """
    Load ThresholdCrossings data array and time axis.
    Returns (data: ndarray T×C, times: ndarray T, rate: float).
    """
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        for obj in nwb.all_children():
            if safe_attr(obj, "name") != "ThresholdCrossings":
                continue
            data = safe_attr(obj, "data", None)
            if data is None:
                continue
            data = np.array(data, dtype=np.float32)      # T × C
            rate  = safe_attr(obj, "rate",          None)
            t0    = safe_attr(obj, "starting_time", 0.0) or 0.0
            ts    = safe_attr(obj, "timestamps",    None)

            if ts is not None:
                times = np.array(ts, dtype=np.float64)
            elif rate is not None and rate > 0:
                times = t0 + np.arange(data.shape[0]) / float(rate)
            else:
                # Fallback: assume 1 sample = 1 ms
                times = t0 + np.arange(data.shape[0]) * 1e-3

            return data, times, float(rate) if rate else 1000.0

    return None, None, None